# درس ۱۳ - حافظه عامل


## راه‌اندازی

این دفترچه نشان می‌دهد چگونه یک نماینده رزرو سفر با **حافظه پایدار** با استفاده از **چارچوب نماینده مایکروسافت** (MAF) بسازید.

شما خواهید آموخت که چگونه انواع مختلف حافظه نماینده — کاری، کوتاه‌مدت و بلندمدت — بر نحوه حفظ و استفاده نماینده از اطلاعات در طول گفتگوها تأثیر می‌گذارند.

**پیش‌نیازها:**
- یک پروژه Azure AI Foundry با یک مدل گفتگو مستقر شده (مثلاً `gpt-4o-mini`).
- وارد شدن با Azure CLI — اجرای دستور `az login` در ترمینال خود.
- `AZURE_AI_PROJECT_ENDPOINT` — نقطه پایانی پروژه Azure AI Foundry شما.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## انواع حافظه عامل

عوامل هوش مصنوعی می‌توانند از انواع مختلف حافظه استفاده کنند که هر کدام هدف متفاوتی دارند:

### حافظه کاری
خود رشته مکالمه — پیام‌های رد و بدل شده در یک جلسه واحد. عامل می‌تواند به پیام‌های قبلی در همان رشته مراجعه کند تا انسجام حفظ شود. در MAF این از طریق **`agent.create_session()`** ایجاد می‌شود که یک `AgentSession` باز می‌گرداند.

### حافظه کوتاه‌مدت
اطلاعاتی که در طول انجام یک کار یا جلسه باقی می‌مانند اما به صورت دائمی ذخیره نمی‌شوند. برای مثال، عامل ممکن است در طول یک مکالمه برنامه‌ریزی چند مرحله‌ای حقایق را جمع‌آوری کند و آن‌ها را برای تولید برنامه نهایی به کار ببرد.

### حافظه بلندمدت
ترجیحات و حقایقی که **در طول جلسات مختلف** باقی می‌مانند. کاربر بازگشتی نباید مجبور باشد محدودیت‌های رژیمی یا سبک سفر خود را تکرار کند. حافظه بلندمدت معمولاً توسط یک ذخیره‌گاه خارجی — یک پایگاه داده، فایل، یا شاخص برداری — پشتیبانی می‌شود و از طریق ابزارها به عامل ارائه می‌گردد.


## حافظه کاری با جلسه‌ها

ساده‌ترین شکل حافظه، جلسه مکالمه است. زمانی که همان جلسه (ایجاد شده توسط `agent.create_session()`) را به تماس‌های متوالی `agent.run()` منتقل می‌کنید، عامل کل تاریخچه آن مکالمه را می‌بیند و می‌تواند جزئیات قبلی را به خاطر بسپارد.

بیایید یک عامل مسافرتی ایجاد کنیم و حافظه کاری را نشان دهیم.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

عامل بودجه را به‌درستی به خاطر آورد زیرا هر دو پیام یک جلسه مشترک داشتند. این همان **حافظه فعال** است — که فقط در طول عمر جلسه وجود دارد.

### با یک رشته جدید چه اتفاقی می‌افتد؟

اگر یک جلسه **جدید** ایجاد کنیم، عامل هیچ خاطره‌ای از مکالمه قبلی ندارد:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## الگوی حافظه بلندمدت

برای به خاطر سپردن ترجیحات کاربر **در جلسات مختلف**، به یک مخزن پایدار نیاز داریم که خارج از رشته مکالمه زندگی کند. عامل (Agent) از طریق **ابزارها** — توابعی که می‌تواند برای ذخیره و بازیابی اطلاعات فراخوانی کند — به این مخزن دسترسی دارد.

در ادامه یک مخزن ساده ترجیحات در حافظه پیاده‌سازی می‌کنیم (در تولید، این را با یک پایگاه داده یا شاخص برداری پشتیبانی می‌کنید) و آن را به‌عنوان ابزارهایی که عامل می‌تواند استفاده کند، در دسترس قرار می‌دهیم.

### معماری
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### سناریو ۱ — کاربر اولین بار سفر سالگرد را رزرو می‌کند

سارا برای اولین بار بازدید می‌کند. نماینده باید ترجیحات او را از طریق ابزارها ذخیره کرده و از آن‌ها برای پیشنهاد هتل‌ها استفاده کند.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### سناریو ۲ — سارا هفته‌ها بعد برمی‌گردد

سارا یک **رشته کاملاً جدید** شروع می‌کند (شبیه‌سازی یک جلسه جدید). حافظه کاری خالی است، اما فروشگاه ترجیحات بلندمدت هنوز اطلاعات او را دارد. عامل باید آن را بازیابی کند و از آن برای شخصی‌سازی پیشنهادها استفاده کند.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصه

در این درس با سه نوع حافظه‌ی عامل و نحوه‌ی پیاده‌سازی آن‌ها با استفاده از چارچوب Microsoft Agent آشنا شدید:

| نوع حافظه | مکانیزم MAF | طول عمر |
|---|---|---|
| **کاری** | `agent.create_session()` | یک مکالمه‌ی واحد |
| **کوتاه‌مدت** | زمینه جمع‌شده در یک رشته | یک کار / جلسه‌ی واحد |
| **بلندمدت** | ذخیره‌سازی خارجی که از طریق توابع `@tool` دسترسی دارد | در چندین جلسه |

### نکات کلیدی
1. **`agent.create_session()`** حافظه کاری را فراهم می‌کند — عامل تاریخچه کامل مکالمه را در یک جلسه می‌بیند.
2. **جلسات جدید زمینه را از دست می‌دهند** — بدون حافظه بلندمدت عامل نمی‌تواند مکالمات گذشته را به خاطر بسپارد.
3. **توابع `@tool` این فاصله را پر می‌کنند** — به عامل اجازه می‌دهند اطلاعات را ذخیره و بازیابی کند از یک ذخیره‌سازی پایدار.
4. **شخصی‌سازی با گذر زمان بهتر می‌شود** — هرچه ترجیحات بیشتر ذخیره شود، توصیه‌های عامل بهتر خواهد بود.

### کاربردهای دنیای واقعی
- **خدمات مشتری**: به خاطر سپردن تاریخچه و ترجیحات مشتری
- **دستیارهای شخصی**: حفظ زمینه در طول روزها یا هفته‌ها
- **مراقبت‌های بهداشتی**: پیگیری اطلاعات و ترجیحات بیمار
- **تجارت الکترونیک**: خرید شخصی‌سازی‌شده بر اساس تاریخچه

### گام‌های بعدی
- جایگزینی دیکشنری در حافظه با پایگاه داده یا فروشگاه برداری (مثلاً Azure AI Search)
- افزودن انقضای حافظه برای اطلاعات حساس به زمان
- ساخت سیستم‌های چندعاملی با حافظه مشترک
- کاوش در نوت‌بوک Cognee برای حافظه مبتنی بر گراف دانش


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:  
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً آگاه باشید که ترجمه‌های خودکار ممکن است دارای خطاها یا نادرستی‌هایی باشند. نسخه اصلی سند به زبان بومی آن باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حساس، استفاده از ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچ گونه سو تفاهم یا سوءتفسیر ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
